# DSCI 551 Project Midterm Report: PostgreSQL Internals
**Focus Area 2: Indexing Mechanisms (Member B)**

This notebook demonstrates progress on the study of PostgreSQL indexing. It covers:
1.  **Dataset Initialization:** Automating the setup of the Asset Tracking system.
2.  **Performance Comparison:** Evaluating Sequential Scan vs. Index Scan.
3.  **Planner Analysis:** Interpreting `EXPLAIN ANALYZE` output to understand cost-based optimization.

In [8]:
import psycopg2
import pandas as pd

# Your verified credentials
db_params = {
    "host": "localhost",
    "database": "postgres",
    "user": "vincentcohen", 
    "password": "" 
}

# Helper function to run queries and return a clean table
def run_query(query):
    conn = psycopg2.connect(**db_params)
    # Using pandas to read the SQL output directly
    df = pd.read_sql(query, conn)
    conn.close()
    return df

print("Environment ready. Connection helper function created.")

Environment ready. Connection helper function created.


In [9]:
# Function to run the external SQL setup file
def initialize_database(file_path):
    try:
        conn = psycopg2.connect(**db_params)
        conn.autocommit = True
        with conn.cursor() as cursor:
            with open(file_path, 'r') as f:
                cursor.execute(f.read())
        print(f"Successfully executed {file_path}")
        conn.close()
    except Exception as e:
        print(f"Error: {e}")

# Run your provided setup script
initialize_database('dataset_setup.sql')

Successfully executed dataset_setup.sql


### Deliverable 1 & 2: Performance Comparison & EXPLAIN Analysis
We first query the `transactions` table (500,000 records) filtering by `user_id`. 
Because no index exists yet on this column, we expect PostgreSQL to use a Sequential Scan.

In [ ]:
query_baseline = "EXPLAIN ANALYZE SELECT * FROM transactions WHERE user_id = 75000;"
df_baseline = run_query(query_baseline)

df_baseline

/var/folders/fv/j_vj6xps6052_k19g06tvl6c0000gn/T/ipykernel_23469/3186591192.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,QUERY PLAN
0,Gather (cost=1000.00..7039.40 rows=2500 width...
1,Workers Planned: 2
2,Workers Launched: 2
3,Buffers: shared hit=3185
4,-> Parallel Seq Scan on transactions (cost...
5,Filter: (user_id = 75000)
6,Rows Removed by Filter: 166666
7,Buffers: shared hit=3185
8,Planning:
9,Buffers: shared hit=70


### Applying a B-Tree Index
We now create a B-tree index on `transactions(user_id)` to observe how the query planner's strategy shifts.

In [11]:
# Create the index using a standard execution
conn = psycopg2.connect(**db_params)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("CREATE INDEX IF NOT EXISTS idx_transactions_userid ON transactions(user_id);")
conn.close()

# Run the query again to see the Index Scan
query_indexed = "EXPLAIN ANALYZE SELECT * FROM transactions WHERE user_id = 75000;"
df_indexed = run_query(query_indexed)

df_indexed

/var/folders/fv/j_vj6xps6052_k19g06tvl6c0000gn/T/ipykernel_23469/3186591192.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,QUERY PLAN
0,Bitmap Heap Scan on transactions (cost=35.80....
1,Recheck Cond: (user_id = 75000)
2,Heap Blocks: exact=3
3,Buffers: shared hit=3 read=3
4,-> Bitmap Index Scan on idx_transactions_us...
5,Index Cond: (user_id = 75000)
6,Index Searches: 1
7,Buffers: shared read=3
8,Planning:
9,Buffers: shared hit=82 read=1


### Midterm Findings: Indexing Efficiency Analysis
Following the experimental design for **Vince/Member B**, we compared point-query performance on the `transactions` table (500,000 records) before and after implementing a B-tree index.

#### 1. Performance Summary Table
| Metric | Baseline (Non-Indexed) | With B-Tree Index | Improvement |
| :--- | :--- | :--- | :--- |
| **Execution Strategy** | Parallel Sequential Scan | Bitmap Index Scan | Strategic Shift |
| **Execution Time** | 34.747 ms | 11.428 ms | ~3x Faster |
| **Buffer Hits (I/O)** | 3,185 (shared hit) | 83 (hit + read) | 97% Reduction |
| **Cost Estimate** | 1000.00..7039.40 | 35.80 (Initial) | 96% Reduction |

#### 2. Technical Observations
* **The "Strategy Shift":** Without an index, PostgreSQL utilized a **Parallel Sequential Scan**, launching 2 workers to scan the entire 500k-row table. [cite_start]After indexing, the planner switched to a **Bitmap Index Scan**, which is specifically efficient for retrieving multiple rows that match a condition.
* **I/O Efficiency (The "Shared Hit" Win):** The most significant find is the reduction in buffer hits from **3,185** to just **83**. [cite_start]This proves that the B-tree index allowed the engine to ignore 97% of the data pages, drastically reducing physical disk/memory pressure[cite: 42].
* **The Planning Trade-off:** Interestingly, the **Planning Time** increased from **2.1 ms** to **181.1 ms**. [cite_start]This illustrates the cost of a "cold" index—the planner had to analyze the newly created B-tree structure for the first time before deciding to use it[cite: 45].
* **Data Scale:** The initialization of the dataset took **51 seconds** to load approximately 900,000 total records. This confirms that bulk loading is faster *before* indexes are created, as the database doesn't have to update the B-tree structure for every single insert.